# Phase 1 — Data Acquisition
## Heart Disease Detection using Phonocardiogram (PCG) and IoT
**BTech Major Project | Data Science | Galgotias College of Engineering & Technology**

**Authors:** Princee Singh (2300971630044) · Priyanshu Kumar (2300971630045)
**Supervisor:** Dr. Kalpna Sagar

---

### Objective
Download both public PCG datasets used in this project, verify their integrity,
back them up to Google Drive, and confirm the label files are readable.

### Datasets Used
| Dataset | Source | Size | Recordings | Labels |
|---|---|---|---|---|
| PhysioNet/CinC Challenge 2016 | physionet.org | ~1.1 GB | ~3,240 | Normal / Abnormal |
| CirCor DigiScope 2022 | physionet.org | ~560 MB | ~5,282 | Murmur Present / Absent / Unknown |

Both datasets are freely available from PhysioNet — no registration required.
They were selected because together they represent two different recording
populations (mixed-age vs. predominantly pediatric), two different label
schemes, and two different sampling rates (2 kHz vs. 4 kHz), which allows
genuine cross-dataset generalisation testing later in the pipeline.

### Why These Two Datasets?
- **PhysioNet 2016** is the standard benchmark for PCG classification — almost
  every paper in this domain reports numbers on it, making our results directly
  comparable to the literature.
- **CirCor 2022** is more recent and includes auscultation-site-level murmur
  annotations (AV/PV/TV/MV per patient), enabling more precise per-recording
  labelling instead of patient-level weak labels.
- Together they provide ~6,500 usable labelled recordings across two distinct
  clinical populations — enough to train a small deep learning model with
  genuine cross-dataset validation.


---
## Step 1 — Mount Google Drive and Create Project Folder Structure

All downloaded data is backed up to Google Drive immediately after downloading,
since Colab's local disk is wiped at the end of every session.
The folder structure created here mirrors the local development environment.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/pcg-project'

folders = [
    'data/raw',
    'data/processed',
    'data/features',
]
for d in folders:
    os.makedirs(os.path.join(PROJECT, d), exist_ok=True)
    print(f"Created: {os.path.join(PROJECT, d)}")

print("\nDrive mounted and folder structure ready.")


---
## Step 2 — Download PhysioNet/CinC Challenge 2016

Downloaded using `wget` with recursive crawl (`-r`), no-clobber (`-N`),
continue (`-c`), and no-parent (`-np`) flags to avoid re-downloading existing
files and to prevent crawling outside the target directory.

**Note on download time:** The first attempt used `-q` (silent mode), which
hid progress output and made the download appear frozen for ~1.5 hours. 
The download was actually completing normally — 7,044 files / 1.2 GB were 
fetched successfully. This is documented here as a reproducibility note: 
omit `-q` or use `--show-progress` to see live progress on future runs.

**Why download to `/content/raw_dl` first?**
Downloading directly to a Drive-mounted path is slow and unreliable for large
numbers of small files (thousands of `.wav`/`.hea` pairs). Downloading to 
Colab's local SSD first, then zipping and copying to Drive in one operation,
is significantly faster and more reliable.


In [ ]:
# Download PhysioNet 2016 to Colab's local disk
# This takes 5-15 minutes depending on network speed
# Progress is shown via --show-progress (remove -q flag from original)
!wget --show-progress -r -N -c -np \
    https://physionet.org/files/challenge-2016/1.0.0/ \
    -P /content/raw_dl


---
## Step 3 — Verify PhysioNet 2016 Download

Check total disk usage and file counts to confirm the download completed
successfully before moving on to CirCor.


In [ ]:
# Check total size and file count
!du -sh /content/raw_dl 2>/dev/null
!find /content/raw_dl -type f | wc -l

# Check audio and header file counts specifically
print("\n--- PhysioNet 2016 file breakdown ---")
!find /content/raw_dl -name "*.wav" | wc -l
!find /content/raw_dl -name "*.hea" | wc -l

# Confirm REFERENCE.csv is present in all 6 training subfolders
print("\n--- REFERENCE.csv locations ---")
!find /content/raw_dl -iname "REFERENCE.csv"


### ✅ Expected Output
- **Total size:** ~1.2 GB
- **Total files:** ~7,000+ (audio + header + label files combined)
- **`.wav` count:** ~3,150 (training-a through training-f, plus validation)
- **`.hea` count:** should match `.wav` count (one header per recording)
- **REFERENCE.csv:** should appear once per subfolder (training-a to training-f)
  plus once in `validation/`

**Observed:** 1.2 GB, 7,044 files, REFERENCE.csv present in all expected
locations. Download verified complete.

> **Note on the `validation/` subfolder:** The recursive wget also pulls in
> the Challenge's original held-out validation set. This subfolder is excluded
> from training/testing in Phase 2 when we build the unified manifest, since
> it was never part of the official training data.


---
## Step 4 — Download CirCor DigiScope 2022

Downloaded using the same recursive wget approach. CirCor is slightly smaller
(~560 MB, ~10,437 files) and downloads faster than PhysioNet 2016.

**Why version 1.0.3?**
Version 1.0.3 is the latest stable release of the CirCor dataset and
includes corrected auscultation-site murmur location annotations compared
to earlier versions. The `Murmur locations` column (which sites the murmur
was audible at) is used in Phase 2 to generate precise per-recording labels
rather than copying the patient-level label to all sites indiscriminately.


In [ ]:
# Download CirCor DigiScope 2022
# This takes 30-60 minutes depending on network speed
!wget --show-progress -r -N -c -np \
    https://physionet.org/files/circor-heart-sound/1.0.3/ \
    -P /content/raw_dl


---
## Step 5 — Verify CirCor 2022 Download


In [ ]:
# Combined file count across both datasets
print("--- Combined dataset verification ---")
print("Total .wav files:")
!find /content/raw_dl -name "*.wav" | wc -l

print("\nLabel files found:")
!find /content/raw_dl -iname "*reference*" -o -iname "*training_data*"


### ✅ Expected Output
- **Total `.wav` files:** ~6,437 (3,150 from PhysioNet 2016 + ~3,287 from CirCor)
- **CirCor stats:** 10,437 files, 560 MB downloaded in ~32 minutes
- **Label files:** REFERENCE.csv in each PhysioNet 2016 subfolder +
  `training_data.csv` at CirCor root

**Observed:** 6,437 `.wav` files total. Both label files found at expected paths.

> **Why CirCor has fewer `.wav` files than expected (~3,287 vs ~5,282):**
> PhysioNet/CirCor 2022 version 1.0.3 contains approximately 60% of the full
> dataset. The remaining ~40% was withheld as the Challenge's private test set
> and has not been publicly released. This is expected and documented behaviour
> — the download is complete, not partial.


---
## Step 6 — Quick Label Preview

Before backing up, confirm the label files are readable and the class
distribution is visible. This is a first look at the class imbalance that will
need to be handled during model training.


In [ ]:
import pandas as pd, glob

# PhysioNet 2016 — peek at one REFERENCE.csv
# Labels: 1 = abnormal, -1 = normal
ref_path = '/content/raw_dl/physionet.org/files/challenge-2016/1.0.0/training-a/REFERENCE.csv'
ref2016 = pd.read_csv(ref_path, header=None, names=['file', 'label'])
print("=== PhysioNet 2016 — training-a label distribution ===")
print(ref2016['label'].map({1: 'abnormal', -1: 'normal'}).value_counts())
print(f"Total in training-a: {len(ref2016)} recordings")

print()

# CirCor 2022 — peek at training_data.csv
circor_path = '/content/raw_dl/physionet.org/files/circor-heart-sound/1.0.3/training_data.csv'
circor = pd.read_csv(circor_path)
print("=== CirCor 2022 — patient-level murmur label distribution ===")
print(circor['Murmur'].value_counts())
print(f"Total patients: {len(circor)}")
print(f"Columns: {circor.columns.tolist()}")


### ✅ Findings — Class Distribution

**PhysioNet 2016 (training-a sample):**
- Normal recordings significantly outnumber abnormal ones
- This imbalance is consistent across all six training subfolders
- Combined across all subfolders: ~2,725 normal / ~816 abnormal (~3.3:1 ratio)

**CirCor 2022:**
- 942 patients total: 695 Absent, 179 Present, 68 Unknown
- The 68 "Unknown" cases (annotators couldn't confidently label) are dropped
  in Phase 2 — they carry no reliable training signal
- Final per-recording class balance after Phase 2 expansion will be roughly 5:1

**Implication for training:** Both datasets are imbalanced toward the "normal"
class, which is realistic (heart disease is less prevalent than healthy hearts in
a general population), but requires explicit handling during model training via
class-weighted loss functions. A naive accuracy metric would be misleading —
a model that always predicts "normal" would score ~80% accuracy without 
learning anything useful. We use AUC-ROC as the primary metric instead.


---
## Step 7 — Backup Both Datasets to Google Drive

Zip each dataset and copy to Drive so future sessions can unzip from Drive
instead of re-downloading from PhysioNet (which takes 30-90 minutes each time).

**Zip strategy:** We zip the full downloaded folder tree, preserving the
relative path structure. This means unzipping to `/content/work` will recreate
the full path `/content/work/content/raw_dl/physionet.org/files/...` — this
nested path behaviour (caused by zipping absolute paths) is known and handled
in Phase 2 by searching recursively for label files rather than hardcoding paths.


In [ ]:
PROJECT = '/content/drive/MyDrive/pcg-project'

print("Zipping PhysioNet 2016...")
!zip -qr /content/physionet2016.zip /content/raw_dl/physionet.org/files/challenge-2016
print("Done.")

print("Zipping CirCor 2022...")
!zip -qr /content/circor2022.zip /content/raw_dl/physionet.org/files/circor-heart-sound
print("Done.")

print("Copying to Drive...")
!cp /content/physionet2016.zip /content/circor2022.zip "{PROJECT}/data/raw/"

print("\nVerifying Drive backup:")
!ls -lh "{PROJECT}/data/raw/"


### ✅ Expected Output
```
total 1.5G
-rw------- 1 root root 1003M  physionet2016.zip
-rw------- 1 root root  451M  circor2022.zip
```

**Observed:** Both zips present in Drive at expected sizes.
PhysioNet 2016: ~1,003 MB | CirCor 2022: ~451 MB | Total: ~1.5 GB

> **Drive storage note:** After Phase 5 (feature extraction), the raw dataset
> zips are safe to delete from Drive — they can always be re-downloaded from
> PhysioNet. At that point only `features_v2.zip` (~3.8 GB) and the small CSV
> manifests need to be retained. This was necessary in practice due to Google
> Drive's 15 GB free storage limit.


---
## Phase 1 Summary

| Step | Action | Status |
|---|---|---|
| 1 | Mount Drive + create folder structure | ✅ |
| 2 | Download PhysioNet/CinC 2016 | ✅ 1.2 GB / 7,044 files |
| 3 | Verify PhysioNet 2016 | ✅ 3,150 .wav + REFERENCE.csv in all 6 subfolders |
| 4 | Download CirCor DigiScope 2022 | ✅ 560 MB / 10,437 files |
| 5 | Verify CirCor 2022 | ✅ 6,437 total .wav files |
| 6 | Label preview + class distribution | ✅ Imbalance confirmed (~3.3:1 and ~5:1) |
| 7 | Backup both datasets to Drive | ✅ 1.5 GB backed up |

### Key Takeaways
- Both datasets downloaded and verified successfully
- Class imbalance (~3-5:1 normal:abnormal) is consistent across both datasets
  and must be handled explicitly during training
- CirCor's per-patient label structure requires expansion to per-recording
  labels in Phase 2, using the `Murmur locations` column for precise site-level
  labelling rather than patient-level weak labels
- The validation subfolder in PhysioNet 2016 was downloaded but will be 
  excluded in Phase 2 — it is the Challenge's original held-out test set

### Next Step → Phase 2: Data Exploration & Unified Manifest Building
In Phase 2 we build a single unified manifest CSV combining both datasets,
perform exploratory analysis of class distributions and audio properties,
and listen to representative samples from each class.
